# Phase 2: Factor Construction — Fama-French 5-Factor Model

## Objective
Construct five systematic risk factors from the cleaned Nifty 500 panel (2011–present)
to explain cross-sectional stock return variation.

---

## Model

$$R_i - R_f = \alpha + \beta_m MKT + \beta_s SMB + \beta_h HML + \beta_r RMW + \beta_c CMA + \varepsilon$$

---

## Factors

| Factor | Full Name | Long Leg | Short Leg | Proxy Variable |
|--------|-----------|----------|-----------|----------------|
| **MKT** | Market Risk Premium | Market portfolio | Risk-free rate | `return - Rf_monthly` |
| **SMB** | Small Minus Big | Small-cap stocks | Large-cap stocks | `market_cap` |
| **HML** | High Minus Low | High B/M (value) | Low B/M (growth) | `bm` = book value / market cap |
| **RMW** | Robust Minus Weak | High profitability | Low profitability | `profitability` = net income / total assets |
| **CMA** | Conservative Minus Aggressive | Low asset growth | High asset growth | `asset_growth` = ΔTotal Assets / Total Assets |

---

## Construction Methodology

### Portfolio Sort — 2×3 Independent Double Sort
For SMB, HML, RMW, and CMA:

- **Size split:** Median `market_cap` → Small (S) / Big (B)
- **Characteristic split:** 30th / 70th percentile → Low / Neutral / High
- **6 portfolios formed:** SL, SN, SH, BL, BN, BH (for each characteristic)
- **Returns:** Value-weighted within each portfolio each month

### Factor Formulas
- ***SMB*** = (1/3)(SH + SN + SL) − (1/3)(BH + BN + BL)  [averaged across HML, RMW, CMA sorts]
- ***HML*** = (1/2)(SH + BH) − (1/2)(SL + BL)
- ***RMW*** = (1/2)(SR + BR) − (1/2)(SW + BW)
- ***CMA*** = (1/2)(SC + BC) − (1/2)(SA + BA)
- ***MKT*** = Σ(w_i × r_i) − Rf   [value-weighted market return minus risk-free rate]

### Rebalancing Convention
- **Annually every July** using prior fiscal year-end fundamentals (FY ends March 31)
- **Why July?** Ensures March fiscal year data is publicly available (3-month lag)
- Size breakpoints recomputed each July on the full eligible universe

### Eligibility Filter (applied each July)
- Positive book value
- Positive market cap
- Non-missing profitability and asset growth
- Minimum 70% monthly price history

---

## Key Assumptions
- **Value-weighted** portfolios (not equal-weighted) — reduces small-stock noise
- **Independent sorts** on size and each characteristic — avoids empty cells
- **No shorting** assumed in Indian context — factor returns are long-short spreads for
  pricing purposes, not tradeable strategies
- Survivorship bias present — delisting returns unavailable for NSE

---

## Output
Five monthly time series: `MKT`, `SMB`, `HML`, `RMW`, `CMA`
saved to `data/processed/ff5_factors.csv`

In [33]:
# ==========================================================
# STEP 0: Imports + Paths
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DIR = DATA_DIR / "processed"


In [34]:
# ==========================================================
# STEP 1: Load Phase 1 Outputs
# ==========================================================

final = pd.read_csv(
    PROCESSED_DIR / "final_panel.csv",
    parse_dates=['date']
)

rf = pd.read_csv(
    PROCESSED_DIR / "rf_monthly.csv",
    parse_dates=['DATE']
)

print(final.shape)

final.head()

(22525, 32)


C:\Users\jayan\AppData\Local\Temp\ipykernel_7912\233255703.py:10: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  rf = pd.read_csv(


,date,symbol,year,month,return,Rf_monthly,market_cap,bm,profitability,asset_growth,...,%dlyqttotradedqty,effective_year,turnover,quarter,net_income,total_assets,source,book_value,index,shares_outstanding
0,2014-06-30,CONCOR,2014,6,0.063979,0.007053,2.897929e+11,2.366518e-08,0.114403,NaN,...,73.59,2013,6.929783e+07,FY,946.0,8269.0,screener_html,6858.0,31.0,2.437180e+08
1,2014-06-30,LICHSGFIN,2014,6,0.028271,0.007053,1.652014e+11,4.591970e-08,0.013759,NaN,...,60.97,2013,1.196196e+09,FY,1319.0,95862.0,screener_html,7586.0,31.0,5.046630e+08
2,2014-06-30,ACC,2014,6,0.098915,0.007053,2.758443e+11,2.979216e-08,0.091626,NaN,...,40.21,2013,3.335800e+08,FY,1162.0,12682.0,screener_html,8218.0,31.0,1.877450e+08
3,2014-06-30,APOLLOHOSP,2014,6,0.096232,0.007053,1.384363e+11,2.150447e-08,0.058879,NaN,...,47.46,2013,1.236280e+08,FY,315.0,5350.0,screener_html,2977.0,31.0,1.391250e+08
4,2014-06-30,PFC,2014,6,0.051892,0.007053,4.053843e+11,6.789114e-08,0.028108,NaN,...,34.97,2013,8.218635e+08,FY,5462.0,194320.0,screener_html,27522.0,31.0,1.320040e+09


In [36]:
rf['Rf_monthly'].describe()

count    172.000000
mean       0.005358
std        0.001468
min        0.000146
25%        0.004633
50%        0.005465
75%        0.006293
max        0.009671
Name: Rf_monthly, dtype: float64

In [3]:
# ==========================================================
# STEP 2: Data Cleaning
# ==========================================================

# Keep only useful columns
cols_needed = [
    'date',
    'symbol',
    'return',
    'Rf_monthly',
    'market_cap',
    'bm',
    'profitability',
    'asset_growth'
]

final = final[cols_needed].copy()


In [4]:
final.isnull().sum()

date              0
symbol            0
return            1
Rf_monthly       81
market_cap        0
bm                0
profitability     0
asset_growth     20
dtype: int64

In [5]:
# Basic Cleaning
final = final.replace(
    [np.inf, -np.inf],
    np.nan
)

# Fix RF
final = final.sort_values('date')

final['Rf_monthly'] = (
    final['Rf_monthly'].ffill()
)

# asset_growth will be fixed later on. First available accounting observation for each stock has no previous value. 
# So the first row becomes NaN. This is correct financially.

# Remove rows with missing returns
final = final.dropna(
    subset=[
        'return'
    ]
)

# Positive market cap only
final = final[
    final['market_cap'] > 0
]

print(final.isnull().sum())

date              0
symbol            0
return            0
Rf_monthly        0
market_cap        0
bm                0
profitability     0
asset_growth     20
dtype: int64


**Canonical FF convention:**

- Portfolios formed every July
- Use previous fiscal-year accounting data
- Hold portfolios from July(t) → June(t+1)

In [6]:
# ==========================================================
# STEP 3: FF Annual Formation Dataset
# ==========================================================

# Calendar fields
final['year'] = final['date'].dt.year
final['month'] = final['date'].dt.month

# FF convention:
# Jan-Jun belong to previous formation cycle
final['formation_year'] = np.where(
    final['month'] >= 7,
    final['year'],
    final['year'] - 1
)

In [7]:
# ----------------------------------------------------------
# JUNE SIZE SNAPSHOT
# ----------------------------------------------------------

june = final[final['month'] == 6].copy()
june = june[['symbol', 'year', 'market_cap']]

june = june.rename(columns={'year': 'formation_year',
                            'market_cap': 'june_market_cap'
                           })

In [8]:
# ----------------------------------------------------------
# MARCH ACCOUNTING SNAPSHOT
# ----------------------------------------------------------

march = final[final['month'] == 3].copy()

march = march[
    [
        'symbol',
        'year',
        'bm',
        'profitability',
        'asset_growth'
    ]
]

# March accounting used for SAME calendar year July formation
march = march.rename(
    columns={
        'year': 'formation_year',
        'bm': 'bm_snapshot',
        'profitability': 'profitability_snapshot',
        'asset_growth': 'asset_growth_snapshot'
    }
)


In [9]:
# ----------------------------------------------------------
# ANNUAL FORMATION DATASET
# ----------------------------------------------------------

formation = june.merge(
    march,
    on=['symbol', 'formation_year'],
    how='inner'
)

print(formation.shape)

formation.head()

(1678, 6)


,symbol,formation_year,june_market_cap,bm_snapshot,profitability_snapshot,asset_growth_snapshot
0,APOLLOHOSP,2015,1.830815e+11,1.562812e-08,0.058879,0.0
1,DIVISLAB,2015,4.990419e+11,6.253407e-09,0.208356,0.0
2,MINDACORP,2015,1.989329e+10,1.768038e-08,0.060213,0.0
3,OIL,2015,2.685876e+11,7.561973e-08,0.084468,0.0
4,LICHSGFIN,2015,2.275526e+11,3.435058e-08,0.013759,0.0


In [10]:
formation['asset_growth_snapshot'].describe()

count    1678.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: asset_growth_snapshot, dtype: float64

In [11]:
final['asset_growth'].describe()

count    22504.000000
mean         0.010146
std          0.066012
min         -0.636156
25%          0.000000
50%          0.000000
75%          0.000000
max          2.410907
Name: asset_growth, dtype: float64

In [12]:
# ==========================================================
# STEP 4: Eligibility Filters
# ==========================================================

In [13]:
# FF5 Eligibility
eligible = formation.copy()

eligible = eligible.replace(
    [np.inf, -np.inf], 
    np.nan)

eligible = eligible.dropna(
    subset=[
        'june_market_cap',
        'bm_snapshot',
        'profitability_snapshot',
        'asset_growth_snapshot'
    ]
)

eligible = eligible[eligible['june_market_cap'] > 0]
eligible = eligible[eligible['bm_snapshot'] > 0]

eligible = eligible.copy()

print("Eligible firms:", len(eligible))

Eligible firms: 1678


In [14]:
# ==========================================================
# STEP 5: Market Factor (MKT)
# ==========================================================

### Market Factor (MKT)

$$
\text{MKT} = R_m - R_f
$$

where:

- **\(R_m\)** = value-weighted market return  
- **\(R_f\)** = risk-free rate

In [15]:
final.columns

Index(['date', 'symbol', 'return', 'Rf_monthly', 'market_cap', 'bm',
       'profitability', 'asset_growth', 'year', 'month', 'formation_year'],
      dtype='object')

In [16]:
final.head()

,date,symbol,return,Rf_monthly,market_cap,bm,profitability,asset_growth,year,month,formation_year
0,2014-06-30,CONCOR,0.063979,0.007053,2.897929e+11,2.366518e-08,0.114403,NaN,2014,6,2013
1,2014-06-30,LICHSGFIN,0.028271,0.007053,1.652014e+11,4.591970e-08,0.013759,NaN,2014,6,2013
2,2014-06-30,ACC,0.098915,0.007053,2.758443e+11,2.979216e-08,0.091626,NaN,2014,6,2013
3,2014-06-30,APOLLOHOSP,0.096232,0.007053,1.384363e+11,2.150447e-08,0.058879,NaN,2014,6,2013
4,2014-06-30,PFC,0.051892,0.007053,4.053843e+11,6.789114e-08,0.028108,NaN,2014,6,2013


In [17]:
# Value Weights
final['weight'] = (
    final['market_cap'] / final.groupby('date')['market_cap'].transform('sum')
)

# Market Return
mkt = (
    final.groupby('date').apply(
        lambda x: np.sum(x['weight'] * x['return'])
    ).reset_index(name='market_return')
)


In [18]:
final.head()

,date,symbol,return,Rf_monthly,market_cap,bm,profitability,asset_growth,year,month,formation_year,weight
0,2014-06-30,CONCOR,0.063979,0.007053,2.897929e+11,2.366518e-08,0.114403,NaN,2014,6,2013,0.106102
1,2014-06-30,LICHSGFIN,0.028271,0.007053,1.652014e+11,4.591970e-08,0.013759,NaN,2014,6,2013,0.060485
2,2014-06-30,ACC,0.098915,0.007053,2.758443e+11,2.979216e-08,0.091626,NaN,2014,6,2013,0.100995
3,2014-06-30,APOLLOHOSP,0.096232,0.007053,1.384363e+11,2.150447e-08,0.058879,NaN,2014,6,2013,0.050686
4,2014-06-30,PFC,0.051892,0.007053,4.053843e+11,6.789114e-08,0.028108,NaN,2014,6,2013,0.148424


In [19]:
mkt.head()

,date,market_return
0,2014-06-30,0.089757
1,2014-07-31,-0.024889
2,2014-08-31,0.062361
3,2014-09-30,0.022555
4,2014-10-31,0.078585


In [20]:
# Merge RF
rf = rf.rename(columns={'DATE': 'date'})

mkt = mkt.merge(rf, on='date', how='left')

In [21]:
# MKT Factor
mkt['MKT'] = (
    mkt['market_return'] - mkt['Rf_monthly']
)

mkt.head()

,date,market_return,Rf_monthly,MKT
0,2014-06-30,0.089757,0.007053,0.082704
1,2014-07-31,-0.024889,0.007306,-0.032195
2,2014-08-31,0.062361,0.007362,0.054999
3,2014-09-30,0.022555,0.007079,0.015476
4,2014-10-31,0.078585,0.007219,0.071366


In [22]:
# ==========================================================
# STEP 6: Helper Function for 2×3 Sorts
# ==========================================================

This function constructs:
- **HML** portfolios
- **RMW** portfolios
- **CMA** portfolios

using:

- Small / Big
- Low / Neutral / High

In [23]:
# Generic 2x3 Sort Function
def assign_2x3_portfolios(df, variable, low_label='L', high_label='H'):

    df = df.copy()
    
    df = df.dropna(subset=[variable, 'june_market_cap'])

    # Size breakpoint (Median)
    size_median = df.groupby('formation_year')['june_market_cap'].transform('median')
    df['SIZE'] = np.where(df['june_market_cap'] <= size_median,'S','B')

    # Characteristic breakpoints
    low = df.groupby('formation_year')[variable].transform(lambda x: x.quantile(0.30))
    high = df.groupby('formation_year')[variable].transform(lambda x: x.quantile(0.70))
    
    # Characteristic Buckets
    df['CHAR'] = 'N'
    df.loc[df[variable] <= low, 'CHAR'] = low_label
    df.loc[df[variable] >= high, 'CHAR'] = high_label

    # Final Portfolio Label
    df['PORT'] = (df['SIZE'] + df['CHAR'])

    return df


In [24]:
# ==========================================================
# STEP 7: HML Construction
# ==========================================================

In [25]:
hml_assign = assign_2x3_portfolios(
    eligible,
    'bm_snapshot',
    low_label='L',
    high_label='H'
)

hml_monthly = final.merge(
    hml_assign[['symbol', 'formation_year', 'PORT']],
    on=['symbol', 'formation_year'],
    how='left'
)

In [26]:
final.head()

,date,symbol,return,Rf_monthly,market_cap,bm,profitability,asset_growth,year,month,formation_year,weight
0,2014-06-30,CONCOR,0.063979,0.007053,2.897929e+11,2.366518e-08,0.114403,NaN,2014,6,2013,0.106102
1,2014-06-30,LICHSGFIN,0.028271,0.007053,1.652014e+11,4.591970e-08,0.013759,NaN,2014,6,2013,0.060485
2,2014-06-30,ACC,0.098915,0.007053,2.758443e+11,2.979216e-08,0.091626,NaN,2014,6,2013,0.100995
3,2014-06-30,APOLLOHOSP,0.096232,0.007053,1.384363e+11,2.150447e-08,0.058879,NaN,2014,6,2013,0.050686
4,2014-06-30,PFC,0.051892,0.007053,4.053843e+11,6.789114e-08,0.028108,NaN,2014,6,2013,0.148424


In [27]:
hml_monthly.head()

,date,symbol,return,Rf_monthly,market_cap,bm,profitability,asset_growth,year,month,formation_year,weight,PORT
0,2014-06-30,CONCOR,0.063979,0.007053,2.897929e+11,2.366518e-08,0.114403,NaN,2014,6,2013,0.106102,NaN
1,2014-06-30,LICHSGFIN,0.028271,0.007053,1.652014e+11,4.591970e-08,0.013759,NaN,2014,6,2013,0.060485,NaN
2,2014-06-30,ACC,0.098915,0.007053,2.758443e+11,2.979216e-08,0.091626,NaN,2014,6,2013,0.100995,NaN
3,2014-06-30,APOLLOHOSP,0.096232,0.007053,1.384363e+11,2.150447e-08,0.058879,NaN,2014,6,2013,0.050686,NaN
4,2014-06-30,PFC,0.051892,0.007053,4.053843e+11,6.789114e-08,0.028108,NaN,2014,6,2013,0.148424,NaN


In [28]:
print(hml_assign['PORT'].value_counts().sort_index())

PORT
BH    185
BL    335
BN    316
SH    322
SL    172
SN    348
Name: count, dtype: int64


In [29]:
hml_assign['PORT'].isnull().sum()

0

In [30]:
print(hml_monthly['PORT'].value_counts().sort_index())

PORT
BH    2118
BL    3874
BN    3654
SH    3689
SL    1957
SN    4008
Name: count, dtype: int64


In [25]:
# Keep only assigned firms
hml_monthly = hml_monthly.dropna(subset=['PORT'])

# ----------------------------------------------------------
# Monthly value-weighted portfolio returns
# ----------------------------------------------------------

hml_ports = (
    hml_monthly
    .groupby(['date', 'PORT'])
    .apply(
        lambda x: np.average(
            x['return'],
            weights=x['market_cap']
        )
    )
    .reset_index(name='ret')
)


In [26]:
# ----------------------------------------------------------
# Pivot portfolios
# ----------------------------------------------------------

hml_pivot = hml_ports.pivot(
    index='date',
    columns='PORT',
    values='ret'
)

# Ensure required portfolios exist
for col in ['SL', 'BL', 'SH', 'BH']:
    if col not in hml_pivot.columns:
        hml_pivot[col] = np.nan

# ----------------------------------------------------------
# HML Factor
# ----------------------------------------------------------

hml_pivot['HML'] = (
    (hml_pivot['SH'] + hml_pivot['BH']) / 2
    -
    (hml_pivot['SL'] + hml_pivot['BL']) / 2
)

hml_factor = (
    hml_pivot[['HML']]
    .reset_index()
)

print(hml_factor.head())

PORT       date       HML
0    2015-07-31 -0.039725
1    2015-08-31 -0.111429
2    2015-09-30  0.091102
3    2015-10-31  0.038417
4    2015-11-30 -0.074944


In [27]:
# ==========================================================
# STEP 8: RMW Construction
# ==========================================================

In [28]:
rmw_assign = assign_2x3_portfolios(
    eligible,
    'profitability_snapshot',
    low_label='W',
    high_label='R'
)

rmw_monthly = final.merge(
    rmw_assign[['symbol', 'formation_year', 'PORT']],
    on=['symbol', 'formation_year'],
    how='left'
)

In [29]:
print(rmw_assign['PORT'].value_counts().sort_index())

PORT
BN    317
BR    304
BW    215
SN    347
SR    203
SW    292
Name: count, dtype: int64


In [30]:
# Keep only assigned firms
rmw_monthly = rmw_monthly.dropna(subset=['PORT'])

# ----------------------------------------------------------
# Monthly value-weighted portfolio returns
# ----------------------------------------------------------

rmw_ports = (
    rmw_monthly
    .groupby(['date', 'PORT'])
    .apply(
        lambda x: np.average(
            x['return'],
            weights=x['market_cap']
        )
    )
    .reset_index(name='ret')
)


In [31]:
# ----------------------------------------------------------
# Pivot portfolios
# ----------------------------------------------------------

rmw_pivot = rmw_ports.pivot(
    index='date',
    columns='PORT',
    values='ret'
)

# Ensure required portfolios exist
for col in ['SW', 'BW', 'SR', 'BR']:
    if col not in rmw_pivot.columns:
        rmw_pivot[col] = np.nan

# ----------------------------------------------------------
# RMW Factor
# ----------------------------------------------------------

rmw_pivot['RMW'] = (
    (rmw_pivot['SR'] + rmw_pivot['BR']) / 2
    -
    (rmw_pivot['SW'] + rmw_pivot['BW']) / 2
)

rmw_factor = (
    rmw_pivot[['RMW']]
    .reset_index()
)

print(rmw_factor.head())

PORT       date       RMW
0    2015-07-31 -0.060547
1    2015-08-31  0.085092
2    2015-09-30 -0.073093
3    2015-10-31 -0.017543
4    2015-11-30 -0.018500


In [32]:
# ==========================================================
# STEP 9: CMA FACTOR (FULLY SELF-CONTAINED)
# ==========================================================

In [31]:
eligible.groupby('formation_year')['asset_growth_snapshot'].describe()

,count,mean,std,min,25%,50%,75%,max
formation_year,,,,,,,,
2015,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016,33.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2017,41.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2018,201.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2019,203.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020,206.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2021,208.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2022,212.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023,220.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
eligible.groupby('formation_year')['asset_growth_snapshot'].nunique()

formation_year
2015    1
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    1
2023    1
2024    1
2025    1
Name: asset_growth_snapshot, dtype: int64

In [33]:
cma_assign = assign_2x3_portfolios(
    eligible,
    'asset_growth_snapshot',
    low_label='C',
    high_label='A'
)

cma_monthly = final.merge(
    cma_assign[['symbol', 'formation_year', 'PORT']],
    on=['symbol', 'formation_year'],
    how='left'
)

In [34]:
print(cma_assign['PORT'].value_counts().sort_index())

PORT
BA    836
SA    842
Name: count, dtype: int64


In [30]:
# Keep only firms assigned to a portfolio
cma_monthly = cma_monthly.dropna(subset=['PORT'])

# ----------------------------------------------------------
# Monthly value-weighted portfolio returns
# ----------------------------------------------------------

cma_ports = (
    cma_monthly
    .groupby(['date', 'PORT'])
    .apply(
        lambda x: np.average(
            x['return'],
            weights=x['market_cap']
        )
    )
    .reset_index(name='ret')
)


In [31]:
# ----------------------------------------------------------
# Pivot to wide format
# ----------------------------------------------------------

cma_pivot = cma_ports.pivot(
    index='date',
    columns='PORT',
    values='ret'
)

# ----------------------------------------------------------
# Ensure all required portfolios exist
# ----------------------------------------------------------

for col in ['SC', 'BC', 'SA', 'BA']:
    if col not in cma_pivot.columns:
        cma_pivot[col] = np.nan

# ----------------------------------------------------------
# CMA Factor
# ----------------------------------------------------------

cma_pivot['CMA'] = (
    (cma_pivot['SC'] + cma_pivot['BC']) / 2
    -
    (cma_pivot['SA'] + cma_pivot['BA']) / 2
)


In [32]:
# ----------------------------------------------------------
# Final CMA series
# ----------------------------------------------------------

cma_factor = (
    cma_pivot[['CMA']]
    .reset_index()
)

print(cma_factor.head())

print("\nPortfolio Counts")
print(cma_monthly['PORT'].value_counts())

print("\nMissing CMA values")
print(cma_factor['CMA'].isna().sum())

PORT       date  CMA
0    2015-07-31  NaN
1    2015-08-31  NaN
2    2015-09-30  NaN
3    2015-10-31  NaN
4    2015-11-30  NaN

Portfolio Counts
PORT
SA    9654
BA    9646
Name: count, dtype: int64

Missing CMA values
129


In [33]:
# ==========================================================
# STEP 10: SMB Construction (Canonical FF5)
# ==========================================================

# ---------- SMB from HML portfolios ----------

hml_pivot['SMB_HML'] = (
    (hml_pivot['SL'] + hml_pivot['SN'] + hml_pivot['SH']) / 3
    -
    (hml_pivot['BL'] + hml_pivot['BN'] + hml_pivot['BH']) / 3
)

for col in ['SL','SN','SH','BL','BN','BH']:
    if col not in hml_pivot.columns:
        hml_pivot[col] = np.nan

# ---------- SMB from RMW portfolios ----------

rmw_pivot['SMB_RMW'] = (
    (rmw_pivot['SW'] + rmw_pivot['SN'] + rmw_pivot['SR']) / 3
    -
    (rmw_pivot['BW'] + rmw_pivot['BN'] + rmw_pivot['BR']) / 3
)

for col in ['SW','SN','SR','BW','BN','BR']:
    if col not in rmw_pivot.columns:
        rmw_pivot[col] = np.nan

# ---------- SMB from CMA portfolios ----------

cma_pivot['SMB_CMA'] = (
    (cma_pivot['SC'] + cma_pivot['SN'] + cma_pivot['SA']) / 3
    -
    (cma_pivot['BC'] + cma_pivot['BN'] + cma_pivot['BA']) / 3
)

for col in ['SC','SN','SA','BC','BN','BA']:
    if col not in cma_pivot.columns:
        cma_pivot[col] = np.nan

KeyError: 'SN'

In [ ]:
# ---------- Merge the three SMB components ----------

smb = (
    hml_pivot[['SMB_HML']]
    .merge(
        rmw_pivot[['SMB_RMW']],
        left_index=True,
        right_index=True,
        how='outer'
    )
    .merge(
        cma_pivot[['SMB_CMA']],
        left_index=True,
        right_index=True,
        how='outer'
    )
)


In [ ]:
# ---------- Canonical SMB ----------

smb['SMB'] = (
    smb['SMB_HML']
    + smb['SMB_RMW']
    + smb['SMB_CMA']
) / 3

smb_factor = smb[['SMB']].reset_index()

print(smb_factor.head())

In [ ]:
# ==========================================================
# STEP 11: Final FF5 Factor Dataset
# ==========================================================

ff5_factors = (
    mkt[['date', 'MKT']]
    .merge(
        smb_factor,
        on='date',
        how='inner'
    )
    .merge(
        hml_factor,
        on='date',
        how='inner'
    )
    .merge(
        rmw_factor,
        on='date',
        how='inner'
    )
    .merge(
        cma_factor,
        on='date',
        how='inner'
    )
    .sort_values('date')
    .reset_index(drop=True)
)

print(ff5_factors.head())

print()

print(ff5_factors.info())

print()

print(ff5_factors.isnull().sum())